# **Prototype 3D-to-2D Craniofacial Shape Analysis for Forensic Identification**

In [ ]:
# Import required libraries
import os
import time
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive
from totalsegmentator.python_api import totalsegmentator

# Mount Google Drive
drive.mount('/content/drive')



In [ ]:

def segmentation_function(input_path: str):
  ### This function segments the nii file from input_path ###

  basename = os.path.basename(input_path).replace(".nii.gz", "").replace(".nii", "")

  # Create the output folder in Volumes
  output_dir_for_segmented = f"/content/drive/MyDrive/Pipeline/Volumes/{basename}"
  os.makedirs(output_dir_for_segmented, exist_ok=True)

  log_file_path = os.path.join(output_dir_for_segmented, "segmentation_log.txt")

  print(f"\nStarting TotalSegmentator for {basename}")
  start_time_str = time.ctime()
  start_time = time.time()
  #Call the craniofacial_structures task
  totalsegmentator(
      input=input_path,
      output=output_dir_for_segmented,
      task="craniofacial_structures",
      output_type="niftis",
      fast=False
  )
  print("Done! Segmented craniofacial files saved to:", output_dir_for_segmented)

  #Call the teeth task
  totalsegmentator(
      input=input_path,
      output=output_dir_for_segmented,
      task="teeth",
      output_type="niftis",
      fast=False
  )
  end_time_str = time.ctime()
  end_time = time.time()
  elapsed_time = end_time - start_time
  print("Done! Segmented teeth files saved to:", output_dir_for_segmented)

  # Write log to file
  with open(log_file_path, 'w') as f:
      f.write(f"Segmentation for: {basename}\n")
      f.write(f"Start Time: {start_time_str}\n")
      f.write(f"End Time: {end_time_str}\n")
      f.write(f"Elapsed Time: {elapsed_time:.2f} seconds\n")
  print(f"Timing log saved to: {log_file_path}")

In [ ]:
def base_projection_function(input_path, output_path):

  #Load the 3D volume
  img = nib.load(input_path)
  data = img.get_fdata()

  mask = (data > 0).astype(np.uint8)

  #Produce projections
  proj_axial   = np.max(mask, axis=2)
  proj_coronal = np.max(mask, axis=1)
  proj_sagittal = np.max(mask, axis=0)

  #Save projections
  out_axial = f"{output_path}/axial_projection.png"
  out_coronal = f"{output_path}/coronal_projection.png"
  # Keep historical spelling: Comparison.ipynb pairs saved Person1/Person2 projections by filename.
  out_sagittal = f"{output_path}/sagital_projection.png"

  plt.imsave(out_axial, proj_axial, cmap='gray')
  plt.imsave(out_coronal, proj_coronal, cmap='gray')
  plt.imsave(out_sagittal, proj_sagittal, cmap='gray')



In [ ]:

def complete_projection_function(data_name: str):
  person = data_name
  #Start the projection
  print(f"Starting projection process for {person}")
  start_pr_time_str = time.ctime()
  start_pr_time = time.time()
  # The volumes file is in Pipeline/Volumes/{person}
  for volume in os.listdir(f'/content/drive/MyDrive/Pipeline/Volumes/{person}'):
    pr_log_file_path = os.path.join(f'/content/drive/MyDrive/Pipeline/Projections/{person}', "projection_log.txt")
    if volume.endswith('.nii.gz') or volume.endswith('.nii'):
      #Create the input link
      input_path = os.path.join(f'/content/drive/MyDrive/Pipeline/Volumes/{person}', volume)
      # Create the output folder
      output_path = os.path.join(f'/content/drive/MyDrive/Pipeline/Projections/{person}', volume.replace('.nii.gz', '').replace('.nii', ''))
      os.makedirs(output_path, exist_ok=True)

      #Run the projection function
      base_projection_function(input_path, output_path)
  end_pr_time_str = time.ctime()
  end_pr_time = time.time()
  elapsed_pr_time = end_pr_time - start_pr_time
  with open(pr_log_file_path, 'w') as f:
    f.write(f"Projection for: {person}\n")
    f.write(f"Start Time: {start_pr_time_str}\n")
    f.write(f"End Time: {end_pr_time_str}\n")
    f.write(f"Elapsed Time: {elapsed_pr_time:.2f} seconds\n")
  print(f"Timing log saved to: {pr_log_file_path}")
  print(f"Finished segment process for {person}")

In [ ]:
def Pipeline():
  # Run through the Data folder; each file is an individual CT volume
  for person in os.listdir('/content/drive/MyDrive/Pipeline/Data'):
    # Check if the data is in the right format
    if person.endswith('.nii.gz') or person.endswith('.nii'):
      print(f"Starting segment process for {person}")
      #Create the input link
      input_path = os.path.join('/content/drive/MyDrive/Pipeline/Data', person)
      #Run the Segmentator for the Data
      segmentation_function(input_path)

      # Complete segmentation
      # Start create projections
      complete_projection_function(person.replace('.nii.gz', '').replace('.nii', ''))





In [ ]:
if __name__ == "__main__":
  Pipeline()